<h1 align="center">Model — Loan Repayment Prediction</h1>

- Build a model that outputs a repayment probability (loan_paid_back) for borrowers in the test set, evaluated on AUC, based on analysis from the EDA notebook.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from itertools import product
from itertools import combinations as comb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
import os
import optuna

/Users/sawadee/pjatk_ds/loan_repayment_predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading Data

In [2]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
orig = pd.read_csv('../data/original.csv')

## Cross-Validation Setup

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SEEDS = [42, 123, 456, 789, 2024, 0, 7, 13, 99, 2137]


## Preprocessing

In [4]:
test_ids = test['id'].copy()

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

- Dropping id to reduce noise

In [5]:
train['loan_paid_back'] = train['loan_paid_back'].astype('int8')
X = train.drop(columns=['loan_paid_back'])
y = train['loan_paid_back']

- Casting the target to int8, then split the training data into features X and target y.

In [6]:
cat_cols = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
for col in cat_cols:
    X[col]    = X[col].astype('category')
    test[col] = test[col].astype('category')

- Casting categorical string columns to pandas category dtype - particularly for LightGBM

## Feature Engineering

In [7]:
grade_order = [
    'A1','A2','A3','A4','A5',
    'B1','B2','B3','B4','B5',
    'C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5',
    'E1','E2','E3','E4','E5',
    'F1','F2','F3','F4','F5'
]
grade_map = {g: i+1 for i, g in enumerate(grade_order)}
X['grade_subgrade']    = X['grade_subgrade'].map(grade_map)
test['grade_subgrade'] = test['grade_subgrade'].map(grade_map)
orig['grade_subgrade'] = orig['grade_subgrade'].map(grade_map)

- Ordinal encoding of `grade_subgrade` — maps A1 -> 1 through F5 -> 30

In [8]:
for k in range(-3, 2):
    X[f'annual_income_d{k}'] = ((X['annual_income'] * 10**k) % 10).astype('int8')
    test[f'annual_income_d{k}'] = ((test['annual_income'] * 10**k) % 10).astype('int8')

for k in range(-3, 1):
    X[f'loan_amount_d{k}'] = ((X['loan_amount'] * 10**k) % 10).astype('int8')
    test[f'loan_amount_d{k}'] = ((test['loan_amount'] * 10**k) % 10).astype('int8')

for k in range(0, 3):
    X[f'interest_rate_d{k}'] = ((X['interest_rate'] * 10**k) % 10).astype('int8')
    test[f'interest_rate_d{k}'] = ((test['interest_rate'] * 10**k) % 10).astype('int8')

X['grade_subgrade_d1'] = ((X['grade_subgrade'] - 1) % 5 + 1).astype('int8')
test['grade_subgrade_d1'] = ((test['grade_subgrade'] - 1) % 5 + 1).astype('int8')

print(f'Features after digit extraction: {X.shape[1]}')

Features after digit extraction: 24


- Extracting individual digits from numerical features at different decimal positions as new features.
- Extracting the subgrade number (1–5) within each letter grade from the ordinal-encoded `grade_subgrade`.

**Note**
- Only those four numerical columns have added a signal, the rest added noise - empirically tested

## Target encodings

In [9]:
# Defining decimal positions for which each feature gets rounded

ROUND_CONFIGS = {
    'annual_income':        [-4, -3],
    'credit_score':         [1, 0],
    'debt_to_income_ratio': [0, -1],
    'loan_amount':          [-3, -2],
    'interest_rate':        [0, -1],
}


# Groups similar values together by rounding, then replaces each group with its average repayment rate — computed OOF to prevent leakage

for col, decimals_list in ROUND_CONFIGS.items():
    for d in decimals_list:
        rounded = X[col].round(d)
        rounded_test = test[col].round(d)
        name = f'TE_{col}_R{d}'
        oof_te = np.zeros(len(X))
        for train_idx, val_idx in skf.split(X, y):
            tmp = y.iloc[train_idx].groupby(rounded.iloc[train_idx]).mean()
            oof_te[val_idx] = rounded.iloc[val_idx].map(tmp).fillna(y.mean())
        X[name] = oof_te
        test[name] = rounded_test.map(y.groupby(rounded).mean()).fillna(y.mean())

print(f'Features after rounding TEs: {X.shape[1]}')

Features after rounding TEs: 34


- Rounds each numerical feature to different levels of precision, then computes a target encoding on each rounded version — coarser granularity produces more stable mean estimates than raw value TEs.

In [10]:
oof_te = np.zeros(len(X))
for train_idx, val_idx in skf.split(X, y):
    mean_target = y.iloc[train_idx].groupby(X['credit_score'].iloc[train_idx]).mean()
    oof_te[val_idx] = X['credit_score'].iloc[val_idx].map(mean_target).fillna(y.mean())
X['credit_score_te'] = oof_te
mean_target_full = y.groupby(X['credit_score']).mean()
test['credit_score_te'] = test['credit_score'].map(mean_target_full).fillna(y.mean())

oof_te = np.zeros(len(X))
for train_idx, val_idx in skf.split(X, y):
    mean_target = y.iloc[train_idx].groupby(X['debt_to_income_ratio'].iloc[train_idx]).mean()
    oof_te[val_idx] = X['debt_to_income_ratio'].iloc[val_idx].map(mean_target).fillna(y.mean())
X['debt_to_income_ratio_te'] = oof_te
mean_target_full = y.groupby(X['debt_to_income_ratio']).mean()
test['debt_to_income_ratio_te'] = test['debt_to_income_ratio'].map(mean_target_full).fillna(y.mean())

- For each unique value of `credit_score` and `debt_to_income_ratio`, replacing it with the average repayment rate of borrowers with that value, computed OOF to prevent leakage.
- These are the two strongest numerical predictors (`debt_to_income_ratio` d=0.889, `credit_score` d=0.602) — target encoding them at exact values captures fine-grained repayment signal that the raw numerical values alone don't fully express for tree-based models.

In [11]:
NUMS = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
for num_col, cat_col in product(NUMS, cat_cols):
    name = f'{num_col}_{cat_col}'
    combined = X[num_col].round(0).astype(str) + '_' + X[cat_col].astype(str)
    oof_te = np.zeros(len(X))
    for train_idx, val_idx in skf.split(X, y):
        tmp = y.iloc[train_idx].groupby(combined.iloc[train_idx]).mean()
        oof_te[val_idx] = combined.iloc[val_idx].map(tmp).fillna(y.mean())
    X[f'TE_{name}'] = oof_te
    tmp_full = y.groupby(combined).mean()
    test_combined = test[num_col].round(0).astype(str) + '_' + test[cat_col].astype(str)
    test[f'TE_{name}'] = test_combined.map(tmp_full).fillna(y.mean())

- Creating target encodings for all 25 numerical×categorical feature combinations, capturing interaction effects between them

In [12]:
es_num = X['employment_status'].cat.codes.astype(float)

for c in ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade']:
    tmp = es_num.groupby(X[c].astype(str)).mean()
    X[f'ME_es_{c}'] = X[c].astype(str).map(tmp).fillna(tmp.mean())
    test[f'ME_es_{c}'] = test[c].astype(str).map(tmp).fillna(tmp.mean())

for c in ['annual_income', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade']:
    tmp = X.groupby(X[c].astype(str))['debt_to_income_ratio'].mean()
    X[f'ME_dti_{c}'] = X[c].astype(str).map(tmp).fillna(tmp.mean())
    test[f'ME_dti_{c}'] = test[c].astype(str).map(tmp).fillna(tmp.mean())

- For each numerical feature, encoding the average `employment_status` code and average `debt_to_income_ratio` at each value — to capture how employment status and debt burden vary across different income/score/loan levels.

In [13]:
for c in ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade']:
    tmp = orig.groupby(c)['loan_paid_back'].mean()
    X[f'TE_ORIG_{c}'] = X[c].map(tmp).fillna(tmp.mean())
    test[f'TE_ORIG_{c}'] = test[c].map(tmp).fillna(tmp.mean())

for c in ['employment_status', 'education_level', 'loan_purpose', 'gender', 'marital_status']:
    tmp = orig.groupby(orig[c].astype(str))['loan_paid_back'].mean()
    X[f'TE_ORIG_{c}'] = X[c].astype(str).map(tmp).fillna(tmp.mean())
    test[f'TE_ORIG_{c}'] = test[c].astype(str).map(tmp).fillna(tmp.mean())

for c in ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade',
          'employment_status', 'education_level', 'loan_purpose', 'gender', 'marital_status']:
    tmp = orig[c].astype(str).value_counts()
    X[f'CE_ORIG_{c}'] = X[c].astype(str).map(tmp).fillna(0)
    test[f'CE_ORIG_{c}'] = test[c].astype(str).map(tmp).fillna(0)

- Target and count encodings from the original dataset, adding real-world repayment signal to the synthetic features.
- The original dataset cannot be used directly for training due to distribution shift, but feature-target relationships are consistent (confirmed in EDA) — making it a valid source for target encodings. Count encodings additionally capture how common each value is in the real-world data.

In [14]:
for c1, c2 in comb(NUMS, 2):
    name = f'{c1}-{c2}'
    combined = X[c1].round(0).astype(str) + '_' + X[c2].round(0).astype(str)
    oof_te = np.zeros(len(X))
    for train_idx, val_idx in skf.split(X, y):
        tmp = y.iloc[train_idx].groupby(combined.iloc[train_idx]).mean()
        oof_te[val_idx] = combined.iloc[val_idx].map(tmp).fillna(y.mean())
    X[f'TE_{name}'] = oof_te
    test_combined = test[c1].round(0).astype(str) + '_' + test[c2].round(0).astype(str)
    test[f'TE_{name}'] = test_combined.map(y.groupby(combined).mean()).fillna(y.mean())

KeyboardInterrupt: 

- Target encodings for all 10 unique pairs of numerical features — concatenates rounded values of each pair and encodes the average repayment rate for that combination, computed OOF to prevent leakage. Captures interactions between numerical pairs that neither feature shows on its own — e.g. high DTI combined with low credit score may carry different risk than each feature predicts independently.

## Model Training

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        objective='binary', metric='auc',
        n_estimators=10000, n_jobs=-1, verbose=-1, # device='gpu' GPU only
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 31, 256),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
        min_child_weight=trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
        bagging_freq=trial.suggest_int('bagging_freq', 1, 3),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    oof = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        m = lgb.LGBMClassifier(**params, random_state=42)
        m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(200), lgb.log_evaluation(-1)])
        oof[val_idx] = m.predict_proba(X_val)[:, 1]
    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(multivariate=True))
study.optimize(objective, n_trials=200, show_progress_bar=True, timeout=5*3600)
print(f'Best OOF AUC: {study.best_value:.4f}')
print('Best params:', study.best_params)
best_lgb_params = study.best_params

- Tunes LightGBM hyperparameters using Optuna TPE sampler across up to 200 trials with a 5-hour timeout — best params are saved to `best_lgb_params` and overwritten by the hardcoded values in the next cell.

In [ ]:
best_lgb_params = {
    'learning_rate': 0.011652255804065733,
    'num_leaves': 51,
    'min_child_samples': 18,
    'min_child_weight': 0.03041896120857024,
    'feature_fraction': 0.6448891577585888,
    'bagging_fraction': 0.7748959995312351,
    'bagging_freq': 2,
    'reg_alpha': 4.403984628880454,
    'reg_lambda': 5.754674877734551
}

- Hardcoded best params from Optuna tuning run on Kaggle (GPU, 5h timeout) — overwrites study result for reproducibility.

In [ ]:
oof_lgb  = np.zeros(len(y))
test_lgb = np.zeros(len(test))

for seed in SEEDS:
    oof_seed  = np.zeros(len(y))
    test_seed = np.zeros(len(test))
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        m_lgb = lgb.LGBMClassifier(
            objective='binary', metric='auc', n_estimators=10000,
            n_jobs=-1, verbose=-1, # device='gpu' GPU only
            random_state=seed, **best_lgb_params
        )
        m_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(200), lgb.log_evaluation(-1)])
        oof_seed[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
        test_seed += m_lgb.predict_proba(test)[:, 1] / 5
    print(f'Seed {seed} AUC: {roc_auc_score(y, oof_seed):.4f}')
    oof_lgb  += oof_seed  / len(SEEDS)
    test_lgb += test_seed / len(SEEDS)

print(f'\nMulti-seed LGB OOF AUC: {roc_auc_score(y, oof_lgb):.4f}')

# Save in case of session interruption
np.save('oof_lgb.npy', oof_lgb)
np.save('test_lgb.npy', test_lgb)

- Trains LightGBM across 10 seeds × 5 folds with early stopping, averaging predictions to reduce variance. OOF and test predictions saved to `.npy` in case of session interruption.

In [ ]:
# X_cb converts categoricals to str because CatBoost requires string categoricals, unlike LightGBM which uses category dtype.

X_cb = X.copy()
test_cb_data = test.copy()
for col in cat_cols:
    X_cb[col]         = X_cb[col].astype(str)
    test_cb_data[col] = test_cb_data[col].astype(str)

def objective_cb(trial):
    params = dict(
        iterations=10000, # task_type='GPU',   GPU only
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        depth=trial.suggest_int('depth', 4, 10),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        bagging_temperature=trial.suggest_float('bagging_temperature', 0.0, 1.0),
        random_strength=trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        border_count=trial.suggest_int('border_count', 32, 255),
        min_data_in_leaf=trial.suggest_int('min_data_in_leaf', 1, 100),
        loss_function='Logloss', eval_metric='AUC',
        random_seed=42, verbose=0, cat_features=cat_cols,
    )
    oof = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X_cb, y):
        X_train, X_val = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        m = CatBoostClassifier(**params)
        m.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=120)
        oof[val_idx] = m.predict_proba(X_val)[:, 1]
    return roc_auc_score(y, oof)

study_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(multivariate=True))
study_cb.optimize(objective_cb, n_trials=200, show_progress_bar=True, timeout=7*3600)
print(f'Best OOF AUC: {study_cb.best_value:.4f}')
print('Best params:', study_cb.best_params)
best_cb_params = study_cb.best_params

- Tunes CatBoost hyperparameters using Optuna TPE sampler across up to 200 trials with a 7-hour timeout — best params are saved to `best_cb_params` and overwritten by the hardcoded values in the next cell.

In [ ]:
best_cb_params = {
    'learning_rate': 0.01603394857975157,
    'depth': 5,
    'l2_leaf_reg': 1.3184110231018757,
    'bagging_temperature': 0.13317493419135865,
    'random_strength': 2.5795296157182164,
    'border_count': 140,
    'min_data_in_leaf': 25
}

- Hardcoded best params from Optuna tuning run on Kaggle — overwrites study result for reproducibility. To re-tune, run the Optuna cell above and remove this cell.

In [ ]:
oof_cb  = np.zeros(len(y))
test_cb = np.zeros(len(test))

for seed in SEEDS:
    oof_seed  = np.zeros(len(y))
    test_seed = np.zeros(len(test))
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb, y)):
        X_train, X_val = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        m_cb = CatBoostClassifier(
            iterations=10000, # task_type='GPU',  GPU only
            random_seed=seed, verbose=0, cat_features=cat_cols,
            **best_cb_params
        )
        m_cb.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=120)
        oof_seed[val_idx] = m_cb.predict_proba(X_val)[:, 1]
        test_seed += m_cb.predict_proba(test_cb_data)[:, 1] / 5
    print(f'Seed {seed} AUC: {roc_auc_score(y, oof_seed):.4f}')
    oof_cb  += oof_seed  / len(SEEDS)
    test_cb += test_seed / len(SEEDS)

print(f'\nMulti-seed CB OOF AUC: {roc_auc_score(y, oof_cb):.4f}')

- Trains CatBoost across 10 seeds × 5 folds with early stopping, averaging predictions to reduce variance and produce stable OOF and test probabilities.

In [ ]:
os.makedirs('../submissions', exist_ok=True)
pd.DataFrame({'id': test_ids, 'loan_paid_back': test_cb}).to_csv('../submissions/submission_cb.csv', index=False)
pd.DataFrame({'id': test_ids, 'loan_paid_back': test_lgb}).to_csv('../submissions/submission_lgb.csv', index=False)

- Saving individual LGB and CatBoost predictions separately — used as safety backup submissions.

## Blending

In [ ]:
best_auc, best_w = 0, 0.5
for w in np.arange(0.05, 0.95, 0.05):
    auc = roc_auc_score(y, w * oof_lgb + (1-w) * oof_cb)
    if auc > best_auc:
        best_auc, best_w = auc, w
print(f'Best simple blend: LGB {best_w:.2f} + CB {1-best_w:.2f} → AUC {best_auc:.4f}')

best_auc_power, best_w_power, best_power = 0, 0.5, 1.0
for w in np.arange(0.05, 0.95, 0.05):
    for power in [1, 2, 3, 4, 5]:
        lgb_rank = rankdata(oof_lgb) / len(oof_lgb)
        cb_rank  = rankdata(oof_cb)  / len(oof_cb)
        auc = roc_auc_score(y, w * lgb_rank**power + (1-w) * cb_rank**power)
        if auc > best_auc_power:
            best_auc_power, best_w_power, best_power = auc, w, power
print(f'Best power blend: LGB {best_w_power:.2f} + CB {1-best_w_power:.2f}, power={best_power} → AUC {best_auc_power:.4f}')

print(f'Solo LGB AUC: {roc_auc_score(y, oof_lgb):.4f}')
print(f'Solo CB AUC:  {roc_auc_score(y, oof_cb):.4f}')

- Searches for the optimal blend weight between LightGBM and CatBoost OOF predictions — first with simple weighted average, then with power rank blending — selecting the combination with highest AUC.

## Submission

In [ ]:
os.makedirs('../submissions', exist_ok=True)

scores = {
    'lgb':    (roc_auc_score(y, oof_lgb), test_lgb),
    'cb':     (roc_auc_score(y, oof_cb),  test_cb),
    'simple': (best_auc, best_w * test_lgb + (1-best_w) * test_cb),
    'power':  (best_auc_power,
               best_w_power * (rankdata(test_lgb)/len(test_lgb))**best_power +
               (1-best_w_power) * (rankdata(test_cb)/len(test_cb))**best_power),
}

best_name = max(scores, key=lambda k: scores[k][0])
best_score, final_pred = scores[best_name]
print(f'Best: {best_name} (AUC {best_score:.4f})')

submission = pd.DataFrame({'id': test_ids, 'loan_paid_back': final_pred})
submission.to_csv('../submissions/submission_best.csv', index=False)
print(f'Rows: {len(submission)}')
print(submission.head())

- Selects the best prediction strategy (solo LGB, solo CB, simple blend, or power blend) based on OOF AUC and saves the final submission.

## Notes for Reproducibility
- Optuna cells (LGB + CB) can be skipped — hardcoded best params are provided in the preceding cells
- `device='gpu'` and `task_type='GPU'` require GPU — change to `'cpu'` and remove `task_type` for local CPU execution
- Full pipeline runtime on GPU (Kaggle): ~8–10h total (LGB Optuna + CB Optuna + multi-seed training)

## Caveat
- OOF AUC is computed on training data folds and serves as a proxy for leaderboard performance — the actual competition score is evaluated on the unseen test set by Kaggle. In practice, local OOF and public leaderboard scores were closely aligned throughout this competition.
- Two final submissions were selected: the best blend (LightGBM + CatBoost) as the primary, and solo LightGBM as a safety backup — which is why individual model predictions are saved separately alongside the blend.

## Results
- Local OOF AUC: 0.9273 (LGB), 0.9273 (CB), 0.9274 (best blend)
- Public leaderboard AUC: 0.92759
- Best strategy: simple 50/50 blend of LGB and CatBoost
- CV setup proved reliable — local OOF and public leaderboard scores were essentially identical